In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, roc_curve, auc
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier

# --- Chargement des données ---
with np.load("/kaggle/input/m2-mias/training_data.npz", allow_pickle=True) as f:
    data = f["data"]
    feature_names = f["feature_labels"]

labels = pd.read_csv("/kaggle/input/m2-mias/training_labels.csv")
y = labels["Label"].values

#  Conversion des données en float (obligatoire pour np.isnan)
data = data.astype(np.float32)

# --- Étape 1 : interpolation temporelle par patient ---
def interpolate_patient_timeseries(data):
    data_interp = np.copy(data)
    for i in range(data.shape[0]):  # Patients
        for j in range(data.shape[2]):  # Features
            series = data[i, :, j]
            mask = ~np.isnan(series)
            if mask.sum() >= 2:
                df_series = pd.Series(series)
                df_series = df_series.interpolate(limit_direction='both')
                data_interp[i, :, j] = df_series
    return data_interp

data_interp = interpolate_patient_timeseries(data)

# --- Étape 2 : masques de présence des valeurs ---
mask = ~np.isnan(data)
mask = mask.astype(float)
X_mask = mask.mean(axis=1)  # (n_samples, 77)

# --- Étape 3 : statistiques globales ---
X_mean = np.nanmean(data_interp, axis=1)
X_std = np.nanstd(data_interp, axis=1)
X_stats = np.concatenate([X_mean, X_std], axis=1)

# --- Étape 4 : pente temporelle ---
def compute_slopes(data):
    n_samples, n_months, n_features = data.shape
    slopes = np.zeros((n_samples, n_features))
    for i in range(n_samples):
        for j in range(n_features):
            series = data[i, :, j]
            months = np.arange(n_months)
            mask = ~np.isnan(series)
            if mask.sum() >= 2:
                fit = np.polyfit(months[mask], series[mask], 1)
                slopes[i, j] = fit[0]
            else:
                slopes[i, j] = 0.0
    return slopes

X_slope = compute_slopes(data_interp)

# --- Fusion finale ---
X_all = np.concatenate([X_stats, X_mask, X_slope], axis=1)  # (n_samples, 77*3)

# --- Imputation + Standardisation ---
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X_all)
X_scaled = scaler.fit_transform(X_imputed)

# --- Train/Test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# --- Modèle XGBoost ---
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
model.fit(X_train, y_train)

# --- Prédiction proba ---
y_proba = model.predict_proba(X_test)[:, 1]

# --- Recherche du meilleur seuil ---
best_threshold = 0.5
best_f1 = 0

print("Seuil\tF1-classe 1\tRecall-classe 1\tPrecision-classe 1")
for threshold in np.arange(0.1, 0.6, 0.05):
    y_pred_thresh = (y_proba >= threshold).astype(int)
    report = classification_report(y_test, y_pred_thresh, output_dict=True, zero_division=0)
    f1 = report['1']['f1-score']
    recall = report['1']['recall']
    precision = report['1']['precision']
    print(f"{threshold:.2f}\t{f1:.4f}\t\t{recall:.4f}\t\t{precision:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

# --- Résultat final ---
print(f"\n Meilleur seuil : {best_threshold:.2f}")
y_pred_final = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_final))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

# --- Courbes ROC et PR ---
fpr, tpr, _ = roc_curve(y_test, y_proba)
precisions, recalls, _ = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc_score(y_test, y_proba):.2f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Courbe ROC")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(recalls, precisions, label=f"PR AUC = {auc(recalls, precisions):.2f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Courbe Precision-Recall")
plt.legend()

plt.tight_layout()
plt.show()

# --- Interprétation SHAP ---
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test, max_display=20)

# --- Chargement des données d’évaluation ---
with np.load("/kaggle/input/m2-mias/evaluation_data.npz", allow_pickle=True) as f_eval:
    data_eval = f_eval["data"]

#  Conversion en float
data_eval = data_eval.astype(np.float32)

# --- Étape 1 : interpolation temporelle ---
data_eval_interp = interpolate_patient_timeseries(data_eval)

# --- Étape 2 : masques de présence des valeurs ---
mask_eval = ~np.isnan(data_eval)
mask_eval = mask_eval.astype(float)
X_mask_eval = mask_eval.mean(axis=1)

# --- Étape 3 : stats globales ---
X_mean_eval = np.nanmean(data_eval_interp, axis=1)
X_std_eval = np.nanstd(data_eval_interp, axis=1)
X_stats_eval = np.concatenate([X_mean_eval, X_std_eval], axis=1)

# --- Étape 4 : pente temporelle ---
X_slope_eval = compute_slopes(data_eval_interp)

# --- Fusion finale ---
X_all_eval = np.concatenate([X_stats_eval, X_mask_eval, X_slope_eval], axis=1)

# --- Imputation + Standardisation identique à l'entraînement ---
X_eval_imputed = imputer.transform(X_all_eval)
X_eval_scaled = scaler.transform(X_eval_imputed)

# --- Prédiction finale avec le meilleur seuil trouvé ---
y_proba_eval = model.predict_proba(X_eval_scaled)[:, 1]
y_pred_eval = (y_proba_eval >= best_threshold).astype(int)

# --- Création du DataFrame pour soumission ---
submission = pd.DataFrame({
    "Id": np.arange(len(y_pred_eval)),
    "Label": y_pred_eval
})




Code Test avec XGBoost + Optuna : f1-score de 0.4850

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
import optuna

# --- Chargement des données ---
with np.load("/kaggle/input/m2-mias/training_data.npz", allow_pickle=True) as f:
    data = f["data"]
    feature_names = f["feature_labels"]

labels = pd.read_csv("/kaggle/input/m2-mias/training_labels.csv")
y = labels["Label"].values

data = data.astype(np.float32)

# --- Fonctions de pré-traitement ---
def interpolate_patient_timeseries(data):
    data_interp = np.copy(data)
    for i in range(data.shape[0]):  # Patients
        for j in range(data.shape[2]):  # Features
            series = data[i, :, j]
            mask = ~np.isnan(series)
            if mask.sum() >= 2:
                df_series = pd.Series(series)
                df_series = df_series.interpolate(limit_direction='both')
                data_interp[i, :, j] = df_series
    return data_interp

def compute_slopes(data):
    n_samples, n_months, n_features = data.shape
    slopes = np.zeros((n_samples, n_features))
    for i in range(n_samples):
        for j in range(n_features):
            series = data[i, :, j]
            months = np.arange(n_months)
            mask = ~np.isnan(series)
            if mask.sum() >= 2:
                fit = np.polyfit(months[mask], series[mask], 1)
                slopes[i, j] = fit[0]
            else:
                slopes[i, j] = 0.0
    return slopes

# --- Pré-traitement complet ---
data_interp = interpolate_patient_timeseries(data)
mask = ~np.isnan(data)
mask = mask.astype(float)
X_mask = mask.mean(axis=1)
X_mean = np.nanmean(data_interp, axis=1)
X_std = np.nanstd(data_interp, axis=1)
X_stats = np.concatenate([X_mean, X_std], axis=1)
X_slope = compute_slopes(data_interp)
X_all = np.concatenate([X_stats, X_mask, X_slope], axis=1)

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

# --- OPTUNA + XGBOOST ---

def objective(trial):
    param = {
        "verbosity": 0,
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "n_jobs": -1,
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }
    
    model = XGBClassifier(**param)
    
    # CV stratifié
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[valid_idx]
        y_tr, y_val = y_train[train_idx], y_train[valid_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        f1_scores.append(f1_score(y_val, preds))
    
    return np.mean(f1_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Meilleurs paramètres Optuna :", study.best_params)
print("Meilleur F1-score Optuna :", study.best_value)

# Entraîner modèle final avec meilleurs paramètres
best_params = study.best_params
model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
model.fit(X_train, y_train)

# Prédictions proba test
y_proba = model.predict_proba(X_test)[:, 1]

# Recherche meilleur seuil
best_threshold = 0.5
best_f1 = 0
print("Seuil\tF1-classe 1\tRecall-classe 1\tPrecision-classe 1")
for threshold in np.arange(0.1, 0.6, 0.05):
    y_pred_thresh = (y_proba >= threshold).astype(int)
    report = classification_report(y_test, y_pred_thresh, output_dict=True, zero_division=0)
    f1 = report['1']['f1-score']
    recall = report['1']['recall']
    precision = report['1']['precision']
    print(f"{threshold:.2f}\t{f1:.4f}\t\t{recall:.4f}\t\t{precision:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nMeilleur seuil : {best_threshold:.2f} avec F1-score {best_f1:.4f}")

y_pred_final = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_final))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

# --- Interprétation SHAP ---
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test, max_display=20)
